In [39]:
import sys
!{sys.executable} -m pip install colour-science numpy pandas plotly matplotlib --quiet
print("All required packages have been installed.")

All required packages have been installed.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\bitap\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [60]:
import numpy as np
import pandas as pd
import colour
import plotly.graph_objects as go
import plotly.express as px

In [59]:
ILLUMINANT_D65 = colour.CCS_ILLUMINANTS['CIE 1931 2 Degree Standard Observer']['D65']
print(f'D65 illuminant: {ILLUMINANT_D65} \nThis is the white point.')


D65 illuminant: [ 0.3127  0.329 ] 
This is the white point.


In [42]:
# To convert RGB to XYZ

def rgb_to_xyz(r, g, b):
    rgb_norm = np.array([r, g, b]) / 255.0
    xyz = colour.RGB_to_XYZ(
        rgb_norm,
        colourspace='sRGB',
        illuminant=ILLUMINANT_D65,
        apply_cctf_decoding=True # Apply gamma correction
    )
    return np.clip(xyz,0, None) # Ensure no negative values

test_colors = {
    'Red': (255, 0, 0),
    'Green': (0, 255, 0),
    'Blue': (0, 0, 255),
    'White': (255, 255, 255),
    'Black': (0, 0, 0),
    'Yellow': (255, 255, 0),
    'Cyan': (0, 255, 255),
    'Magenta': (255, 0, 255)
}

print(f'{"Colour":<30} {"X":>8} {"Y":>8} {"Z":>8}')
for name, (r, g, b) in test_colors.items():
    xyz = rgb_to_xyz(r, g, b)
    print(f'{name:<30} {xyz[0]:>8.4f} {xyz[1]:>8.4f} {xyz[2]:>8.4f}')

print()    
print('Note that white should have the highest Y value ~ 1.0, while black should have all values close to zero.')

Colour                                X        Y        Z
Red                              0.4124   0.2126   0.0193
Green                            0.3576   0.7152   0.1192
Blue                             0.1805   0.0722   0.9505
White                            0.9505   1.0000   1.0890
Black                            0.0000   0.0000   0.0000
Yellow                           0.7700   0.9278   0.1385
Cyan                             0.5381   0.7874   1.0697
Magenta                          0.5929   0.2848   0.9698

Note that white should have the highest Y value ~ 1.0, while black should have all values close to zero.


In [43]:
# To convert RGB to Lab

def rgb_to_lab(r, g, b):
    xyz = rgb_to_xyz(r, g, b)
    lab = colour.XYZ_to_Lab(xyz, illuminant=ILLUMINANT_D65)
    return lab

print(f'{"Colour":<30} {"L*":>8} {"a*":>8} {"b*":>8}')
for name, (r, g, b) in test_colors.items():
    lab = rgb_to_lab(r, g, b)
    print(f'{name:<30} {lab[0]:>8.2f} {lab[1]:>8.2f} {lab[2]:>8.2f}')
    
print()
print('Red has high a* (reddish) and positive b* (yellowish-red hue).')
print('Blue has negative b* (bluish) as expected.')

Colour                               L*       a*       b*
Red                               53.23    80.11    67.22
Green                             87.74   -86.18    83.19
Blue                              32.30    79.20  -107.85
White                            100.00     0.01     0.00
Black                              0.00     0.00     0.00
Yellow                            97.14   -21.55    94.49
Cyan                              91.12   -48.08   -14.12
Magenta                           60.32    98.26   -60.83

Red has high a* (reddish) and positive b* (yellowish-red hue).
Blue has negative b* (bluish) as expected.


In [44]:
# To convert RGB to HSV

def rgb_to_hsv(r, g, b):
    rgb_norm = np.array([r, g, b]) / 255.0
    h, s, v = colour.RGB_to_HSV(rgb_norm)
    return float(h * 360), float(s), float(v)

print(f'{"Colour":<30} {"Hue (°)":>10} {"Saturation":>12} {"Value":>8}')
for name, (r, g, b) in test_colors.items():
    h, s, v = rgb_to_hsv(r, g, b)
    print(f'{name:<30} {h:>10.1f} {s:>12.2f} {v:>8.2f}')
    
print()
print('White and Black both have Saturation=0 (no colour, just lightness).')
print('Pure Red is at Hue=0°, pure Green at 120°, pure Blue at 240°.')

Colour                            Hue (°)   Saturation    Value
Red                                   0.0         1.00     1.00
Green                               120.0         1.00     1.00
Blue                                240.0         1.00     1.00
White                                 0.0         0.00     1.00
Black                                 0.0         0.00     0.00
Yellow                               60.0         1.00     1.00
Cyan                                180.0         1.00     1.00
Magenta                             300.0         1.00     1.00

White and Black both have Saturation=0 (no colour, just lightness).
Pure Red is at Hue=0°, pure Green at 120°, pure Blue at 240°.


In [45]:
# xy chromaticity coordinates

def rgb_to_xy(r, g, b):
    xyz = rgb_to_xyz(r, g, b)
    total = xyz.sum()
    if total == 0:
        return 0.3127, 0.329
    return float(xyz[0] / total), float(xyz[1] / total)

print(f'{"Color":<30} {"x":>8} {"y":>8}')
for name, (r, g, b) in test_colors.items():
    x, y = rgb_to_xy(r, g, b)
    print(f'{name:<30} {x:>8.4f} {y:>8.4f}')
    
print()
print('D65 white point: x=0.3127, y=0.3290 (centre of the diagram).')

Color                                 x        y
Red                              0.6401   0.3300
Green                            0.3000   0.6000
Blue                             0.1500   0.0600
White                            0.3127   0.3290
Black                            0.3127   0.3290
Yellow                           0.4193   0.5053
Cyan                             0.2247   0.3287
Magenta                          0.3209   0.1542

D65 white point: x=0.3127, y=0.3290 (centre of the diagram).


In [46]:
# plot the chromaticity diagram

cmfs = colour.MSDS_CMFS['CIE 1931 2 Degree Standard Observer']
XYZ_locus = cmfs.values
total_locus = XYZ_locus.sum(axis=1)
mask = total_locus > 0
locus_x = np.where(mask, XYZ_locus[:, 0] / total_locus, 0)
locus_y = np.where(mask, XYZ_locus[:, 1] / total_locus, 0)

locus_x_closed = np.append(locus_x, locus_x[0])
locus_y_closed = np.append(locus_y, locus_y[0])


srgb = colour.RGB_COLOURSPACES['sRGB']
pri = srgb.primaries
tri_x = list(pri[:, 0]) + [pri[0, 0]]
tri_y = list(pri[:, 1]) + [pri[0, 1]]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=locus_x_closed, y=locus_y_closed,
    mode='lines', line=dict(color='black', width=2),
    name='Spectral locus'
))

fig.add_trace(go.Scatter(
    x=tri_x, y=tri_y,
    mode='lines', line=dict(color='black', dash='dot', width=1.5),
    name='sRGB Gamut'
))

fig.add_trace(go.Scatter(
    x=[ILLUMINANT_D65[0]], y=[ILLUMINANT_D65[1]],
    mode='markers+text',
    marker=dict(color='white', size=10, line=dict(color='black', width=1)),
    text=['D65'], textposition='top right',
    name='D65 white point'
))

colour_map = {
    'Red'     : '#FF0000',
    'Green'     : '#00FF00',
    'Blue'     : '#0000FF',
    'White' : '#AAAAAA',
    'Black'       : '#444444',
    'Yellow'  : "#F2FF00",
    'Cyan'    : "#00EAFF",
    'Magenta' : "#BF00FF"
}


for name, (r, g, b) in test_colors.items():
    cx, cy = rgb_to_xy(r, g, b)
    short_name = name.split('(')[0].strip()
    fig.add_trace(go.Scatter(
        x=[cx], y=[cy],
        mode='markers+text',
        marker=dict(color=colour_map[name], size=12, line=dict(color='black', width=1)),
        text=[short_name], textposition='top right',
        name=short_name, showlegend=False
    ))
    
    
fig.update_layout(
    title='CIE 1931 xy Chromaticity Diagram for test colors',
    xaxis=dict(title='x', range=[0, 0.85], dtick=0.1),
    yaxis=dict(title='y', range=[0, 0.90], dtick=0.1, scaleanchor='x'),
    width=620, height=560,
)
     
fig.show()               
                    

In [49]:
# color difference

def delta_e_2000(lab1, lab2):
    return float(colour.delta_E(lab1, lab2, method='CIE 2000'))

def delta_e_94(lab1, lab2):
    return float(colour.delta_E(lab1, lab2, method='CIE 1994'))


# Compare the pairs
colour_names  = list(test_colors.keys())
colour_values = list(test_colors.values())

print('Color difference between different pairs \n')

print(f'{"Pair":<55} {"ΔE 2000":>10} {"ΔE 94":>10}')

for i in range(len(colour_names)):
    for j in range (i + 1, len(colour_names)): # This loops 
        lab_a = rgb_to_lab(*colour_values[i])
        lab_b = rgb_to_lab(*colour_values[j])
        de2000 = delta_e_2000(lab_a, lab_b)
        de94   = delta_e_94(lab_a, lab_b)
        pair_name = f"{colour_names[i].split('(')[0].strip()} vs {colour_names[j].split('(')[0].strip()}"
        print(f'{pair_name:<55} {de2000:>10.2f} {de94:>10.2f}')


Color difference between different pairs 

Pair                                                       ΔE 2000      ΔE 94
Red vs Green                                                 86.61      73.43
Red vs Blue                                                  52.88      70.57
Red vs White                                                 45.81      50.23
Red vs Black                                                 50.41      56.30
Red vs Yellow                                                64.31      60.00
Red vs Cyan                                                  70.96      67.60
Red vs Magenta                                               42.59      50.70
Green vs Blue                                                83.18     105.90
Green vs White                                               33.27      22.41
Green vs Black                                               87.87      89.72
Green vs Yellow                                              23.40      24.18
Green vs Cyan        

In [50]:
# Now we try it for one of our samples that is goniochromatic, and we choose two angles that show different colors.

COL_A = (104, 56, 24)  # Cham020, 60/30
COL_B = (44, 244, 202) # Cham020, 25/21


lab_a = rgb_to_lab(*COL_A)
lab_b = rgb_to_lab(*COL_B)
de    = delta_e_2000(lab_a, lab_b)

def interpret_de(de):
    if de < 1:   return 'Imperceptible'
    elif de < 2: return 'Perceptible to trained observers only'
    elif de < 3.5: return 'Perceptible at a glance'
    elif de < 5: return 'Clearly different'
    else:        return 'Distinctly different'

print('Color A:', COL_A)
print(f'  L*={lab_a[0]:.2f}  a*={lab_a[1]:.2f}  b*={lab_a[2]:.2f}')
print()
print('Color B:', COL_B)
print(f'  L*={lab_b[0]:.2f}  a*={lab_b[1]:.2f}  b*={lab_b[2]:.2f}')
print()
print(f'ΔE CIE 2000 = {de:.3f}')
print(f'Interpretation: {interpret_de(de)}')


Color A: (104, 56, 24)
  L*=29.00  a*=18.46  b*=28.53

Color B: (44, 244, 202)
  L*=86.75  a*=-55.96  b*=7.08

ΔE CIE 2000 = 69.151
Interpretation: Distinctly different


In [ ]:
# Now we work on spectral data

df_spectra = pd.read_csv('sample_data/sample_spectra.csv')

print(f'Shape: {df_spectra.shape[0]} rows x {df_spectra.shape[1]} columns')
print(f'Wavelength range: {df_spectra["wavelength_nm"].min()} - {df_spectra["wavelength_nm"].max()} nm')
print()
print('First few rows: ')
df_spectra.head()

Shape: 41 rows x 8 columns
Wavelength range: 380 - 780 nm

First few rows: 


,wavelength_nm,Red_Vermillion,Green_Emerald,Blue_Cobalt,Yellow_Chrome,White_Titanium,Grey_Neutral_50,Black_Carbon
0,380,0.05,0.04,0.20,0.05,0.92,0.5,0.04
1,390,0.05,0.04,0.28,0.05,0.92,0.5,0.04
2,400,0.05,0.04,0.38,0.05,0.92,0.5,0.04
3,410,0.05,0.05,0.48,0.05,0.92,0.5,0.04
4,420,0.05,0.07,0.55,0.05,0.92,0.5,0.04


In [53]:
curve_colors = {
    'Red_Vermillion'  : '#cc2200',
    'Green_Emerald'   : '#00aa77',
    'Blue_Cobalt'     : '#0044cc',
    'Yellow_Chrome'   : '#ddaa00',
    'White_Titanium'  : '#aaaaaa',
    'Grey_Neutral_50' : '#666666',
    'Black_Carbon'    : '#222222',
}


wavelengths = df_spectra['wavelength_nm'].values
sample_cols = [c for c in df_spectra.columns if c != 'wavelength_nm']

fig = go.Figure()

for col in sample_cols:
    fig.add_trace(go.Scatter(
        x=wavelengths,
        y=df_spectra[col].values,
        mode='lines',
        name=col.replace('_', ' '),
        line=dict(color=curve_colors.get(col, '#888'), width=2)
    ))
    
fig.update_layout(
    title='Spectral Reflectance Curves for Reference Pigments',
    xaxis=dict(title='Wavelength (nm)', range=[380, 780]),
    yaxis=dict(title='Reflectance', range=[0, 1.05]),
    width=750, height=450,
    legend=dict(x=1.01, y=0.99)
)
fig.show()

In [54]:
cmfs_2deg   = colour.MSDS_CMFS['CIE 1931 2 Degree Standard Observer']
illuminant_sd = colour.SDS_ILLUMINANTS['D65']


def spectra_to_xyz(wavelengths, reflectance):
    sd = colour.SpectralDistribution(dict(zip(wavelengths.tolist(), reflectance.tolist())))
    xyz = colour.sd_to_XYZ(sd, cmfs=cmfs_2deg, illuminant=illuminant_sd) / 100.0
    return np.clip(xyz, 0, None)

def spectra_to_lab(wavelengths, reflectance):
    xyz = spectra_to_xyz(wavelengths, reflectance)
    return colour.XYZ_to_Lab(xyz, ILLUMINANT_D65)

def spectra_to_rgb_display(wavelengths, reflectance):
    xyz = spectra_to_xyz(wavelengths, reflectance)
    rgb = colour.XYZ_to_RGB(xyz, colourspace='sRGB',
                             illuminant=ILLUMINANT_D65, apply_cctf_encoding=True)
    rgb = np.clip(rgb, 0, 1)
    return tuple(int(round(c * 255)) for c in rgb)

print(f'{"Sample":<20} {"L*":>8} {"a*":>8} {"b*":>8}  {"sRGB (approx)"}')
for col in sample_cols:
    ref = df_spectra[col].values.astype(float)
    lab = spectra_to_lab(wavelengths, ref)
    r, g, b = spectra_to_rgb_display(wavelengths, ref)
    print(f'{col:<20} {lab[0]:>8.2f} {lab[1]:>8.2f} {lab[2]:>8.2f}  ({r:3d},{g:3d},{b:3d})')


Sample                     L*       a*       b*  sRGB (approx)
Red_Vermillion          41.12    57.06    24.79  (184, 39, 60)
Green_Emerald           56.12   -68.25    -6.39  (  0,161,144)
Blue_Cobalt             27.72    55.71   -71.67  ( 70, 27,179)
Yellow_Chrome           83.15    16.71    94.15  (255,192,  0)
White_Titanium          96.82     0.00     0.01  (246,246,246)
Grey_Neutral_50         76.07     0.00     0.01  (188,188,188)
Black_Carbon            23.67     0.00     0.00  ( 56, 56, 56)


In [55]:
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=locus_x_closed, y=locus_y_closed,
    mode='lines', line=dict(color='black', width=2), name='Spectral locus'
))

fig2.add_trace(go.Scatter(
    x=tri_x, y=tri_y,
    mode='lines', line=dict(color='grey', dash='dot', width=1.5), name='sRGB gamut'
))

fig2.add_trace(go.Scatter(
    x=[ILLUMINANT_D65[0]], y=[ILLUMINANT_D65[1]],
    mode='markers+text',
    marker=dict(color='white', size=10, line=dict(color='black', width=1)),
    text=['D65'], textposition='top right', name='D65'
))

for col in sample_cols:
    ref = df_spectra[col].values.astype(float)
    xyz = spectra_to_xyz(wavelengths, ref)
    total = xyz.sum()
    cx = float(xyz[0] / total) if total > 0 else 0.3127
    cy = float(xyz[1] / total) if total > 0 else 0.3290
    r, g, b = spectra_to_rgb_display(wavelengths, ref)
    hex_col = f'#{r:02X}{g:02X}{b:02X}'
    fig2.add_trace(go.Scatter(
        x=[cx], y=[cy],
        mode='markers+text',
        marker=dict(color=hex_col, size=14, line=dict(color='black', width=1.5)),
        text=[col.replace('_', ' ')], textposition='top right',
        name=col.replace('_', ' '), showlegend=True
    ))

fig2.update_layout(
    title='CIE 1931 xy Chromaticity — Spectral Reference Samples',
    xaxis=dict(title='x', range=[0, 0.85], dtick=0.1),
    yaxis=dict(title='y', range=[0, 0.90], dtick=0.1, scaleanchor='x'),
    width=650, height=580,
)

fig2.show()

In [56]:
# Color difference between two samples -- Feel free to change the samples
SAMPLE_A = 'Red_Vermillion'
SAMPLE_B = 'Yellow_Chrome'

ref_a = df_spectra[SAMPLE_A].values.astype(float)
ref_b = df_spectra[SAMPLE_B].values.astype(float)

lab_a = spectra_to_lab(wavelengths, ref_a)
lab_b = spectra_to_lab(wavelengths, ref_b)

de_2000 = delta_e_2000(lab_a, lab_b)
de_94   = delta_e_94(lab_a, lab_b)

print(f'Comparing: {SAMPLE_A}  vs  {SAMPLE_B}')
print()
print(f'{SAMPLE_A:<20}  L*={lab_a[0]:.2f}  a*={lab_a[1]:.2f}  b*={lab_a[2]:.2f}')
print(f'{SAMPLE_B:<20}  L*={lab_b[0]:.2f}  a*={lab_b[1]:.2f}  b*={lab_b[2]:.2f}')
print()
print(f'ΔE CIE 2000  = {de_2000:.3f}')
print(f'ΔE CIE 94    = {de_94:.3f}')
print(f'Interpretation: {interpret_de(de_2000)}')

Comparing: Red_Vermillion  vs  Yellow_Chrome

Red_Vermillion        L*=41.12  a*=57.06  b*=24.79
Yellow_Chrome         L*=83.15  a*=16.71  b*=94.15

ΔE CIE 2000  = 55.270
ΔE CIE 94    = 57.175
Interpretation: Distinctly different


In [57]:
# Now we compute all the color differences

lab_values = {}
for col in sample_cols:
    ref = df_spectra[col].values.astype(float)
    lab_values[col] = spectra_to_lab(wavelengths, ref)

n = len(sample_cols)
de_matrix = np.zeros((n, n))

for i, a in enumerate(sample_cols):
    for j, b in enumerate(sample_cols):
        de_matrix[i, j] = delta_e_2000(lab_values[a], lab_values[b])

short_names = [c.replace('_', ' ') for c in sample_cols]
df_de = pd.DataFrame(de_matrix, index=short_names, columns=short_names)
df_de = df_de.round(2)

print('ΔE CIE 2000 matrix:')
print()
df_de

ΔE CIE 2000 matrix:



,Red Vermillion,Green Emerald,Blue Cobalt,Yellow Chrome,White Titanium,Grey Neutral 50,Black Carbon
Red Vermillion,0.00,74.32,36.91,55.27,50.96,40.96,29.78
Green Emerald,74.32,0.00,55.85,53.93,39.97,31.68,39.41
Blue Cobalt,36.91,55.85,0.00,86.00,66.05,56.38,30.00
Yellow Chrome,55.27,53.93,86.00,0.00,31.52,30.73,65.18
White Titanium,50.96,39.97,66.05,31.52,0.00,13.45,64.12
Grey Neutral 50,40.96,31.68,56.38,30.73,13.45,0.00,52.39
Black Carbon,29.78,39.41,30.00,65.18,64.12,52.39,0.00


In [58]:
fig_heatmap = px.imshow(
    de_matrix,
    x=short_names,
    y=short_names,
    color_continuous_scale='RdYlGn_r',  
    text_auto='.1f',
    title='ΔE CIE 2000 Heatmap for Pairwise Colour Differences',
    labels=dict(color='ΔE 2000')
)
fig_heatmap.update_layout(width=620, height=560)
fig_heatmap.show()

